# RAG Evaluation

## Ground Truth

Ground truth data was generated with `data/generate_ground_truth.py` script. The script makes a call to LLM for each of the FAQ records to generate 5 sample questions users might ask that a given record anwsers. This dataset will serve as a base for evaluation of retrieval and generation steps 

In [1]:
from pathlib import Path
import pandas as pd

data_path = Path().resolve().parent / "data" / "csv" / "ground_truth.csv"
ground_truth = pd.read_csv(data_path)
ground_truth.head(10)

,id,question
0,1,What is GeForce NOW and how does it work?
1,1,Can you explain what NVIDIA GeForce NOW is?
2,1,Is GeForce NOW a cloud gaming service?
3,1,What devices can I use GeForce NOW on?
4,1,How does GeForce NOW let me play the games I own?
5,2,What countries or regions is GeForce NOW avail...
6,2,"Is GeForce NOW available in North America, Eur..."
7,2,Where can I use GeForce NOW right now?
8,2,Does GeForce NOW work outside North America an...
9,2,Which locations support GeForce NOW through NV...


In [3]:
from elasticsearch import Elasticsearch
from constants import DEFAULT_ES_URL, DEFAULT_INDEX, DEFAULT_VECTOR_INDEX

client = Elasticsearch(DEFAULT_ES_URL)
if not client.ping():
    raise RuntimeError(f"Cannot connect to Elasticsearch at {DEFAULT_ES_URL}")

## Retrieval evaluation

To evaluate different search approaches with different parameters we calculate following metrics : **Hit Rate**, **MRR**. Functions for metrics calculation and evaluation function were prepared in `games_assistant/evaluation_utils.py` model. Evaluation for different approaches is run against the whole **Ground Truth** dataset - evaluation checks if relevant records are returned for ground truth questions.

### Boost parameters for text search

First we run evaluation for text search to find the best values for boost parameters (`boost_dict` that contains weights for searchable fields in text index). Evaluation metrics are calculated and compared for different combinations of boost parameters for fields: **question**, **tag** and **anwser**.

In [4]:
from itertools import product

from evaluation_utils import evaluate_search_function
from faq_text_search import search_faq


boost_values = [1, 2, 3, 5]
boost_combinations = product(boost_values, repeat=3)

evaluation_results = []
for question_boost, tag_boost, answer_boost in boost_combinations:
    boost_dict = {
        "question": question_boost,
        "tag": tag_boost,
        "answer": answer_boost,
    }
    metrics = evaluate_search_function(
        search_function=search_faq,
        ground_truth=ground_truth,
        index_name=DEFAULT_INDEX,
        client=client,
        size=5,
        boost_dict=boost_dict,
    )
    evaluation_results.append({
        "question_boost": question_boost,
        "tag_boost": tag_boost,
        "answer_boost": answer_boost,
        **metrics,
        "average": (metrics["hit_rate"] + metrics["mrr"]) / 2,
    })

evaluation_results = (
    pd.DataFrame(evaluation_results)
    .sort_values(["average", "hit_rate", "mrr"], ascending=False)
    .reset_index(drop=True)
)
evaluation_results.head(10)

,question_boost,tag_boost,answer_boost,hit_rate,mrr,average
0,3,2,3,0.942857,0.819218,0.881037
1,5,3,5,0.940816,0.819388,0.880102
2,2,1,2,0.940816,0.818367,0.879592
3,3,1,3,0.940816,0.818367,0.879592
4,5,1,5,0.940816,0.818367,0.879592
5,5,2,5,0.940816,0.818367,0.879592
6,1,1,1,0.942857,0.815238,0.879048
7,2,2,2,0.942857,0.815238,0.879048
8,3,3,3,0.942857,0.815238,0.879048
9,5,5,5,0.942857,0.815238,0.879048


The following boost parameters give the best results: `{"question": 3, "tag": 2, "answer": 3}`

### Text Search vs Vector Search vs Hybrid Search

### Document Reranking ??